# Data Lake - Jogos Olímpicos

## Fontes de Dados

Os dados utilizados neste projeto foram obtidos a partir das seguintes fontes públicas:
* [Base dos Dados – Histórico das Olimpíadas](https://www.kaggle.com/datasets/piterfm/paris-2024-olympic-summer-games)(`raw/athlete_event_result.csv`, `raw/athlete_bio.csv` e `raw/country.csv`)
* [Paris 2024 Olympic Summer Games Dataset](https://basedosdados.org/dataset/62f8cb83-ac37-48be-874b-b94dd92d3e2b)(`raw/athletes.csv`, `raw/medals.csv` e `raw/nocs.csv`)

In [39]:
import pandas as pd

## Leitura dos dados brutos

In [40]:
results = pd.read_csv("raw/athlete_event_result.csv")
bio = pd.read_csv("raw/athlete_bio.csv")
country = pd.read_csv("raw/country.csv")

athletes_2024 = pd.read_csv("raw/athletes.csv")
medals_2024 = pd.read_csv("raw/medals.csv")
nocs_2024 = pd.read_csv("raw/nocs.csv")

C:\Users\anafl\AppData\Local\Temp\ipykernel_632\4171043409.py:1: DtypeWarning: Columns (0: medal) have mixed types. Specify dtype option on import or set low_memory=False.
  results = pd.read_csv("raw/athlete_event_result.csv")


In [41]:
for df in [results, bio, country, athletes_2024, medals_2024, nocs_2024]:
    df.columns = df.columns.str.lower().str.strip()

## Integração dos dados históricos

In [42]:
print([bio.columns])
print([results.columns])
print([country.columns])

[Index(['athlete_id', 'name', 'sex', 'birth_date', 'birth_year', 'height',
       'weight', 'country', 'country_noc', 'description', 'special_notes'],
      dtype='str')]
[Index(['edition', 'edition_id', 'country_noc', 'sport', 'event', 'result_id',
       'athlete', 'athlete_id', 'position', 'medal', 'is_team_sport'],
      dtype='str')]
[Index(['noc', 'name'], dtype='str')]


In [43]:
historico = results.merge(bio[['athlete_id', 'sex']], on='athlete_id', how='left')
historico = historico.merge(country[['noc', 'name']], left_on='country_noc', right_on='noc', how='left')

historico = historico.rename(columns={"name": "country"})

historico = historico[['sex', 'country', 'sport', 'medal']]
historico = historico[historico['medal'].notna()]

historico.head()

,sex,country,sport,medal
273023,Male,United States,Skeleton,Gold
273024,Male,Italy,Skeleton,Gold
273025,Male,United States,Skeleton,Gold
273026,Female,United States,Skeleton,Gold
273027,Male,Canada,Skeleton,Gold


## Integração dos dados Paris 2024

In [44]:
paris = medals_2024.merge(nocs_2024[['code', 'country']], left_on='country_code', right_on='code', how='left')
paris = paris.merge(athletes_2024[['name', 'gender']], on='name', how='left')

paris = paris.rename(columns={"gender_x": "sex", "country_y": "country", "discipline": "sport", "medal_type": "medal"})

paris = paris[['sex', 'name', 'country', 'sport', 'medal']]

paris.head()

,sex,name,country,sport,medal
0,M,Remco EVENEPOEL,Belgium,Cycling Road,Gold Medal
1,M,Filippo GANNA,Italy,Cycling Road,Silver Medal
2,M,Wout van AERT,Belgium,Cycling Road,Bronze Medal
3,W,Grace BROWN,Australia,Cycling Road,Gold Medal
4,W,Anna HENDERSON,Great Britain,Cycling Road,Silver Medal


## União dos datasets

In [45]:
paris = medals_2024.merge(
    nocs_2024[['code', 'country']],
    left_on='country_code',
    right_on='code',
    how='left'
)

paris = paris.merge(
    athletes_2024[['name', 'gender']],
    on='name',
    how='left'
)

In [46]:
olimpiadas = pd.concat([historico, paris], ignore_index=True)
olimpiadas = olimpiadas[olimpiadas['medal'].notna()]
olimpiadas = olimpiadas[olimpiadas['country'].notna()]
olimpiadas = olimpiadas[olimpiadas['sport'].notna()]